# Aralin 05 - Agentic RAG


## Setup

Ipinapakita ng notebook na ito ang Agentic RAG (Retrieval-Augmented Generation) na pattern gamit ang Microsoft Agent Framework.

**Mga Kinakailangan:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — ang iyong Azure AI Search service endpoint
- `AZURE_SEARCH_API_KEY` — ang iyong Azure AI Search API key
- Naka-configure ang Azure OpenAI deployment sa pamamagitan ng mga environment variable
- Na-authenticate ang Azure CLI (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Ano ang Agentic RAG?

Ang Tradisyonal na RAG ay sumusunod sa isang nakapirming proseso: kumuha ng mga dokumento, pagkatapos ay gumawa ng tugon. **Ang Agentic RAG** ay mas higit pa sa pamamagitan ng pagbibigay sa ahente ng kalayaan kung **kailan** at **paano** kukuha ng impormasyon.

Sa Agentic RAG, ang ahente ay maaaring:
- **Magpasya** kung kailangan ba ng pagkuha bago sumagot sa isang tanong
- **Pumili** kung aling pinagmumulan ng data o kasangkapan ang tatanungin
- **Suriin** ang mga nakuha na resulta at magsagawa ng karagdagang pagkuha kung hindi sapat ang unang pagtatangka
- **Pagsamahin** ang impormasyon mula sa maraming hakbang ng pagkuha upang makabuo ng isang magkakaugnay na sagot

Ito ay ginagawang mas flexible at tumpak ang ahente kumpara sa isang statikong proseso na kumuha muna, tapos gumawa ng sagot.


## Paglikha ng Kasangkapan sa Paghahanap

Sa Agentic RAG, ang mga panlabas na pinagkukunan ng datos ay iniimpake bilang **mga kasangkapan** na maaaring tawagin ng ahente kapag kailangan. Pinapayagan nito ang ahente na ituring ang pagkuha ng impormasyon bilang isa pang aksyon na maaari nitong gawin, sa halip na isang mandatoryong hakbang.

Sa ibaba ay tinutukoy namin ang isang knowledge base tungkol sa paglalakbay at inilalantad ito bilang isang kasangkapan na maaaring tawagin ng ahente upang maghanap ng impormasyon tungkol sa destinasyon.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## Paggawa ng RAG Agent

Ngayon ay gagawa tayo ng isang agent na inuutusan **na palaging kumuha ng impormasyon bago sumagot**. Ginagamit ng agent ang tool na `search_travel_knowledge` upang iugnay ang mga sagot nito sa database ng kaalaman kaysa umasa sa sariling training data nito.


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## Iterative Retrieval — Ang Maker-Checker Pattern

Isang pangunahing bentahe ng Agentic RAG ay ang **iterative retrieval**. Maaaring magsagawa ang agent ng maraming ulit na paghahanap upang beripikahin, pinuhin, o palawakin ang mga paunang natuklasan — katulad ng isang "maker-checker" na daloy ng trabaho:

1. **Hakbang ng Maker**: Kinukuha ng agent ang paunang impormasyon at gumagawa ng draft ng sagot.
2. **Hakbang ng Checker**: Nagsasagawa ang agent ng karagdagang paghahanap upang beripikahin ang mga detalye o punan ang mga kakulangan.

Sa ibaba, tinanong ang agent ng isang tanong na nangangailangan ng paghahambing ng iba't ibang destinasyon, kaya't pinipilit itong magsagawa ng maraming paghahanap.


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## Buod

Sa araling ito, natutunan mo kung paano bumuo ng isang **Agentic RAG** na sistema gamit ang Microsoft Agent Framework:

- Pinapayagan ng **Agentic RAG** ang mga ahente na awtonomong magpasya kung kailan kukuha ng impormasyon, kaya ang pagkuha ay nagiging dinamiko sa halip na nakatakda.
- **Mga tools bilang pinanggagalingan ng datos**: Ang mga panlabas na kaalaman (tulad ng Azure AI Search) ay ginawang mga tool na maaaring gamitin ng ahente.
- **Iteratibong pagkuha**: Pinapahintulutan ng maker-checker pattern ang ahente na magsagawa ng maraming pag-ikot ng pagkuha — paghahanap, pag-verify, at pag-aayos — bago magbigay ng pinal na sagot.

Sa produksyon, papalitan mo ang in-memory na `TRAVEL_KNOWLEDGE_BASE` ng totoong Azure AI Search index para hawakan ang malakihang pagkuha ng mga dokumento sa paglalakbay.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Paunawa**:
Ang dokumentong ito ay isinalin gamit ang AI translation service na [Co-op Translator](https://github.com/Azure/co-op-translator). Bagamat aming pinagsisikapang maging tumpak, pakatandaan na ang mga awtomatikong pagsasalin ay maaaring maglaman ng mga pagkakamali o hindi pagkakatugma. Ang orihinal na dokumento sa orihinal nitong wika ang dapat ituring na opisyal na sanggunian. Para sa mga mahahalagang impormasyon, inirerekomenda ang propesyonal na pagsasaling pantao. Hindi kami mananagot sa anumang hindi pagkakaunawaan o maling interpretasyon na maaaring magmula sa paggamit ng pagsasaling ito.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
